In [94]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn # To ignore some warnings from sklearn logistic regression


from datetime import datetime
import numpy as np
from scipy.optimize import minimize # minimizing function for AIOLI
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
import pandas as pd
from sklearn.linear_model import LogisticRegression # for classic logistic regression
from typing import Literal

options={"disp": False} # To not get verbose from minimize function

## Functions for algorithms

First I define some functions that will be used later in all algorithms.

In [32]:
def sig(z):
    """ Sigmoid function that avoids overflow """
    if z < 0:
        return np.exp(z) / (1 + np.exp(z))
    else:
        return 1 / (1 + np.exp(-z))

In [33]:
# NOTE TO DO : put in general algorithm function that if radius == NA then this is not done.
def project_onto_l2_ball(x,radius):
    """ Euclidean projection onto L2 ball"""
    norm = np.linalg.norm(x)
    if norm <= radius:
        return x
    return x * radius/norm

In [34]:
def calculate_OTB(array, method: Literal["suffix_averaging", "uniform", "last_iterate"] | None = None, p = 0.5):
    """ Weighted average function
    ----
    Parameters:
        p: Proportion of last observations to average over. Ex p = 0.3 means average over last 30% of observations.
    Methods:
        Suffix averaging: Average over last p percentage of observations.
        Uniform: Average over all iterates equally
        Last iterate: Consider only the last iterate.
        """
    n = len(array)
    # Check p is a proportion
    if p is None:
        p = 0.5
    if not (0 <= p <= 1):
        raise ValueError("p must be between 0 and 1")

    if method == "suffix_averaging":
        i = round((1-p)*n)
        result = np.mean(array[i:], axis=0)
    elif method == "uniform" or method is None:
        result = np.mean(array, axis=0)
    elif method == "last_iterate":
        last_iterate = n - 1
        result = array[last_iterate]
    else:
        raise ValueError("Method must be one of the options")
    return result

In [35]:
def log_loss(betas,data_x,data_y):
    return sum(np.logaddexp(0,-data_y*np.inner(data_x,betas)))

In [36]:
def generate_y(n,x, betas, linearly_dependent_on_x = True, all_x = True, independent_method = 1,sinusoidal = False, quadratic = False, seed = 40):
    y = np.zeros(n)
    rng = np.random.default_rng(seed) # Random number generator

    if linearly_dependent_on_x and all_x:
        if sinusoidal:
            for t in range(n):
                prob_y1 = (1/2)*np.sin(4*np.inner(x[t],betas)) + 1/2
                y[t] = rng.choice([1,-1], size = 1, p = [prob_y1, 1-prob_y1]).item()
        elif quadratic:
            for t in range(n):
                prob_y1 = sig(np.inner(x[t]**2,betas))
                y[t] = rng.choice([1,-1], size = 1, p = [prob_y1, 1-prob_y1]).item()
        else:   # following sigmoid function
            for t in range(n):
                prob_y1 = sig(np.inner(x[t],betas))
                y[t] = rng.choice([1,-1], size = 1, p = [prob_y1, 1-prob_y1]).item()
    elif linearly_dependent_on_x:
        x1_x2 = x[:,0:2]
        for t in range(n):
            prob_y1 = sig(np.inner(x1_x2[t],betas[0:2]))
            y[t] = rng.choice([1,-1], size = 1, p = [prob_y1, 1-prob_y1]).item()
    elif sinusoidal:
        for t in range(n):
            prob_y1 = np.sin(np.inner(x[t],betas))
            y[t] = rng.choice([1,-1], size = 1, p = [prob_y1, 1-prob_y1]).item()
    else:
        if independent_method == 1:
            prob_y1 = rng.uniform(0.1,0.9)
            y = rng.choice([1,-1], size = n, p = [prob_y1, 1-prob_y1])
        if independent_method == 2:
            for t in range(n):
                prob_y1 = sig(np.inner((1/x[t]),betas))
                y[t] = rng.choice([1,-1], size = 1, p = [prob_y1, 1-prob_y1]).item()
        if independent_method == 3:
            # Unbalanced classes
            prob_y1 = 0.8
            y = rng.choice([1,-1], size = n, p = [prob_y1, 1-prob_y1])

    return y

In [37]:
def generate_x(n,d, correlated = True, means = None, sds = None, highly_correlated = True, seed = 40):
    rng = np.random.default_rng(seed) # Random number generator
    if sds is None:
        sds = rng.uniform(0.1,4,d)
    if means is None:
        means = [rng.normal(1,sd,1).item() for sd in sds]

    if not correlated:
        cov = np.diag(np.zeros(d))
        x = np.random.multivariate_normal(means, cov, size = n)
    if correlated:
        diagonal = rng.uniform(9,10,d)
        cov = np.diag(diagonal)
        x = np.random.multivariate_normal(means, cov, size = n)
        if highly_correlated and d == 3:
            x1 = generate_x(n,1)
            x2 = x1 + rng.normal(2,1)
            x3 = x1 + x2
            x = np.concatenate((x1,x2,x3), axis = 1)

    return x


# Online Gradient Descent

In [40]:
def online_gd(n, d, data_y, data_x, B, OTB_method = None, OTB_average_proportion = None):
    """Online gradient descent
    Parameters
    ---------
    n : float
        Size of the dataset
    d : float
        Dimension of the context vector, i.e. number of independent variables
    data_y : ndarray
        Vector with outcome(y) values. Dimensions: n x 1
    data_x: ndarray
        Array with independent variables. Dimensions: n x d
    eta: float
        Learning rate.
    B: float
        Radius of the L2 ball constraint for the parameter vector beta
    OTB_method: string
        Averaging method, default none which implies the averaging function default which is uniform.
    OTB_average_proportion:
    """

    # Start time
    start_time = datetime.now()

    # Initialize vectors
    betas = np.zeros((n,d)) # Parameter vector of all n rounds.

    loss = np.zeros(n)
    G = 1 # Max gradient
    eta = 1
    gradient = np.zeros(d)
    y_hat = np.zeros(n)

    for t in range(n):
        # Observe context vector for this round
        xt = data_x[t]

        # Update parameter (beta)
        if t == 0:
            beta_tilde = np.zeros(d)
        else:
            beta_tilde = betas[t-1] - eta*gradient

        # Euclidean projection of beta onto the L2 ball
        #betas[t] = project_onto_l2_ball(beta_tilde,B)
        betas[t] = beta_tilde

        # Issue prediction
        y_hat[t] = sig(np.inner(betas[t],xt))

        # Observe true y from the data
        yt = data_y[t]

        # Calculate gradient and sum of gradients
        gradient = -yt*xt*sig(-yt*np.inner(xt,betas[t]))
        # update maximum gradient
        G = np.max([G,np.linalg.norm(gradient)**2])

        # Suffer loss
        loss[t] = np.logaddexp(0,-yt*np.inner(xt,betas[t]))

        # Recalculate eta
        eta = 1 / np.sqrt((t+1) * G) # Here t+1 simply because python initializes t to 0


    # Store endtime
    end_time = datetime.now()

    # Get Batch estimate
    betas_OTB = calculate_OTB(betas, OTB_method, OTB_average_proportion)

    # Get Batch estimate loss
    loss_OTB =  log_loss(betas_OTB,data_x, data_y)

    return{
        "runtime" : (end_time - start_time).total_seconds(),
        "Total_Loss" : loss_OTB,
        "Online_Loss" : sum(loss),
        "Batch_estimate": betas_OTB,
        "All_Betas": betas,
        "predictions": y_hat
    }


## Testing ODG

In [41]:
#ogd = online_gd(n = 40,d = 3,data_y = y,data_x = x,B = 40)

# AIOLI

In [71]:
class AIOLI:
    def __init__(self):
        self.first_sum  = []
        self.etas_gradients = []
        self.x0 = [] # Initializer for minimizing algorithm
        self.reg_lambda = 0
        self.train_size = 0
        self.d = 0
        self.n = 0

    # Define function to minimize for AIOLI
    @staticmethod # Because I dont use the self
    def parameter_function_AIOLI(b, xt, reg_lambda, first_sum, etas_gradients):
        l_hat = np.inner(b,first_sum) + b @ etas_gradients @ b #  @ to do matrix-vector multiplication
        # Use function np.logaddexp = log(exp(x1) + exp(x2)) to ensure numerical stability, not sure how it works
        result = l_hat + np.logaddexp(0,np.inner(b,xt)) + np.logaddexp(0,np.inner(-b,xt)) + reg_lambda*np.linalg.norm(b)**2
        return result
    def debug(self):
        print(self.etas_gradients)

    def new_point_loss(self,X,Y, OTB_method):
        loss_new = np.zeros(self.train_size)
        convergence_new = 0
        beta_hat = np.zeros((self.train_size,self.d))
        first_sum  = np.zeros(self.d)
        etas_gradients = np.zeros((d,d))
        for t in range(self.train_size):
            # Minimize using data up until t-1
            result = minimize(self.parameter_function_AIOLI, self.x0, method = "l-bfgs-b", args=(X, self.reg_lambda,first_sum, etas_gradients))
            beta_hat[t] = result.x
            convergence_new += result.success
            # Update minimization elements for next round
            first_sum += self.first_sum[t]
            etas_gradients += self.etas_gradients[t]

        # Get OTB
        big_beta = calculate_OTB(beta_hat, OTB_method)

        # Suffer loss
        loss_point = np.logaddexp(0,-Y*np.inner(X,big_beta))

        return loss_point

    def new_data_loss(self,new_data_x,new_data_y, OTB_method: Literal["suffix_averaging", "uniform", "last_iterate"] | None = None):
        start_time = datetime.now()
        new_losses = [ self.new_point_loss(X,Y, OTB_method) for (X,Y) in zip(new_data_x,new_data_y) ] # Zip pairs the rows of x and y
        end_time = datetime.now()
        return{
                "runtime" : (end_time - start_time).total_seconds(),
                "loss" : sum(new_losses)
            }


    def fit(self, data_y, data_x, B, X, OTB_method = None, OTB_average_proportion = None, minimize_method = "l-bfgs-b"):
        """AIOLI
        This is the function to run AIOLI.

        Parameters
        ---------
        data_y : ndarray
            Vector with outcome(y) values. Dimensions: n x 1
        data_x: ndarray
            Array with independent variables. Dimensions: n x d
        B: float
            Radius of the L2 ball constraint for the parameter vector beta
        X: float
            Radius of the L2 ball constraint for the feature vector x"""
        # Record start time
        start_time = datetime.now()

        # Get data dimensions
        self.n = data_x.shape[0]
        n = self.n
        self.d = data_x.shape[1]
        d = self.d
        self.x0 = np.zeros(d) # Initializer for minimizing algorithm
        self.train_size = n
        # Initialize vectors for storage
        betas = np.zeros((n, d))
        ys = np.zeros(n)
        y_hats = np.zeros(n)
        loss = np.zeros(n)
        etas = np.zeros(n)
        gradients = np.zeros((n, d))
        self.reg_lambda = 1 / (B**2)

        # Keep sums for loss functions
        self.first_sum  = np.zeros((n, d))
        self.etas_gradients = np.zeros((n,d,d))
        first_sum  = np.zeros(d)
        etas_gradients = np.zeros((d,d))

        # Track convergence
        convergence = 0
        minimizer_gradient = 0

        # Run Algorithm
        for t in range(n):
            # Get context vector
            xt = data_x[t]

            # Update beta parameter
            # noinspection PyTypeChecker # This is to avoid the underlining which
            if t == 0: # Initialize to 0 for first round
                beta_tilde = np.zeros(d)
            else:
                result = minimize(self.parameter_function_AIOLI,
                                  np.zeros(d),
                                  method = minimize_method,
                                  args=(xt, self.reg_lambda,first_sum, etas_gradients),
                                  options = {"maxiter":100000}
                                  )
                betas[t] = result.x
                convergence += result.success
                minimizer_gradient += np.linalg.norm(result.jac)

            # Project beta onto set
            #betas[t] = project_onto_l2_ball(beta_tilde,B)
            #betas[t] = self.x0

            # Generate prediction
            y_hats[t] = np.inner(xt,betas[t])

            # Observe true y from data
            ys[t] = data_y[t]

             # Compute gradient
            gradients[t] = -ys[t]*xt*sig(-ys[t]*np.inner(xt,betas[t]))

            # Estimate eta
            etas[t] = np.exp(ys[t]*y_hats[t])/(1+B*X)

            # Suffer loss
            loss[t] = np.logaddexp(0,-ys[t]*np.inner(xt,betas[t]))

            # Compute elements for AIOLI loss function

            fs = gradients[t]*(1 - etas[t]*np.inner(betas[t],gradients[t]))
            eg = (etas[t] / 2) * np.outer(gradients[t], gradients[t])
            self.first_sum[t] = fs
            self.etas_gradients[t] = eg
            first_sum += fs
            etas_gradients += eg
            # Here this is separated and a bit reiterative because else the way the vector is stored messes up with the function. I tried multiple ways and this seemed the best one.
        # for loop ends

        # Get Batch estimate
        betas_OTB = calculate_OTB(betas, OTB_method, OTB_average_proportion)

        # Get Batch estimate loss
        loss_OTB = log_loss(betas_OTB,data_x, data_y)

        end_time = datetime.now()

        return{
            "runtime" : (end_time - start_time).total_seconds(),
            "Total_Loss" : loss_OTB,
            "Online_Loss" : sum(loss),
            "Batch_estimate": betas_OTB,
            "All_Betas": betas,
            "convergence" : convergence,
            "minimizer_gradient" : minimizer_gradient
        }



## Testing AIOLI

In [67]:
x = generate_x(100,3)
betas= [3,0.1,5]
y = generate_y(100,x,betas = betas)

x_train = x[:80]
y_train = y[:80]
x_test = x[80:]
y_test = y[80:]

## AIOLI convergence check

In [68]:
aioli_model = AIOLI()
aioli_results = aioli_model.fit(data_y = y_train, data_x = x_train, X = 10, B = 40, minimize_method = "l-bfgs-b")
aioli_results

{'runtime': 0.260098,
 'Total_Loss': np.float64(51.84154755149404),
 'Online_Loss': np.float64(9.190996781488307),
 'Batch_estimate': array([ 45.37676398, -18.6927405 ,  26.68407416]),
 'All_Betas': array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [-2.74677920e+01,  9.40815439e+01,  6.66137492e+01],
        [ 1.49609155e+02, -1.28959589e+02,  2.06469231e+01],
        [ 5.65103481e+01,  1.15184132e+01,  6.80289394e+01],
        [ 6.23386753e+01, -5.88586435e+01,  3.47993221e+00],
        [ 6.49986514e+01, -3.37190550e+01,  3.12797373e+01],
        [ 1.21009841e+02, -9.14377623e+01,  2.95688904e+01],
        [ 7.73500002e+01, -5.90580279e+01,  1.82922362e+01],
        [ 8.15102815e+01, -6.23215448e+01,  1.91922597e+01],
        [ 1.21813259e+02, -9.24826450e+01,  2.93310552e+01],
        [ 6.73969273e+01, -5.05452936e+01,  1.68526256e+01],
        [ 6.69164336e+01, -3.52086827e+01,  3.17069445e+01],
        [ 6.91573707e+00,  3.93179469e+00,  1.08464993e+01],
        [

In [69]:
minimization_methods = ["l-bfgs-b",'bfgs',"CG"]

for method in minimization_methods:
    aioli_model = AIOLI()
    aioli_results = aioli_model.fit(data_y = y_train, data_x = x_train, X = 10, B = 40, minimize_method = method)
    print(f"Method: {method}")
    print(f"convergence: {aioli_results["convergence"]}")
    print(f"minimizer_gradient: {aioli_results["minimizer_gradient"]}")
    print(f"Online loss: {aioli_results["Online_Loss"]}")


Method: l-bfgs-b
convergence: 79
minimizer_gradient: 0.0016883095589787219
Online loss: 9.190996781488307
Method: bfgs
convergence: 76
minimizer_gradient: 0.0006533526719269104
Online loss: 9.190993277305651
Method: CG
convergence: 75
minimizer_gradient: 0.0007800553514449521
Online loss: 9.191000473624506


In [70]:
aioli_model.new_data_loss(new_data_x = x_test, new_data_y = y_test, OTB_method = "uniform")

{'runtime': 5.949214, 'loss': np.float64(1.5683564551461988)}

# Ada Grad

In [51]:
# OJO STILL HAVE TO CODE THIS. the code is just OGD.
def online_AdaGrad(n, d, data_y, data_x, B, OTB_method = None, OTB_average_proportion = None):
    """Online Adaptive Gradient
    Parameters
    ---------
    n : float
        Size of the dataset
    d : float
        Dimension of the context vector, i.e. number of independent variables
    data_y : ndarray
        Vector with outcome(y) values. Dimensions: n x 1
    data_x: ndarray
        Array with independent variables. Dimensions: n x d
    eta: float
        Learning rate.
    B: float
        Radius of the L2 ball constraint for the parameter vector beta
    OTB_method: string
        Averaging method, default none which implies the averaging function default which is uniform.
    OTB_average_proportion:
    """

    # Start time
    start_time = datetime.now()

    # Initialize vectors
    betas = np.zeros((n,d)) # Parameter vector of all n rounds.

    loss = np.zeros(n)
    G = 1 # Max gradient
    eta = 1
    gradient = np.zeros(d)

    for t in range(n):
        # Observe context vector for this round
        xt = data_x[t]

        # Update parameter (beta)
        if t == 0:
            beta_tilde = np.zeros(d)
        else:
            beta_tilde = betas[t-1] - eta*gradient

        # Euclidean projection of beta onto the L2 ball
        #betas[t] = project_onto_l2_ball(beta_tilde,B)
        betas[t] = beta_tilde

        # Issue prediction
        #y_hat = sig(np.inner(betas[t],xt)) commented out because its never used so there's no need to calculate it

        # Observe true y from the data
        yt = data_y[t]

        # Calculate gradient and sum of gradients
        gradient = -yt*xt*sig(-yt*np.inner(xt,betas[t]))
        # update maximum gradient
        G = np.max([G,np.linalg.norm(gradient)**2])

        # Suffer loss
        loss[t] = np.logaddexp(0,-yt*np.inner(xt,betas[t]))

        # Recalculate eta
        eta = 1 / np.sqrt((t+1) * G) # Here t+1 simply because python initializes t to 0


    # Store endtime
    end_time = datetime.now()

    # Get Batch estimate
    betas_OTB = calculate_OTB(betas, OTB_method, OTB_average_proportion)

    # Get Batch estimate loss
    loss_OTB =  log_loss(betas_OTB,data_x, data_y)

    return{
        "runtime" : (end_time - start_time).total_seconds(),
        "Total_Loss" : loss_OTB,
        "Online_Loss" : sum(loss),
        "Batch_estimate": betas_OTB,
        "All_Betas": betas
    }


## Logistic Regression

In [52]:
def classic_logistic_regression(data_x,data_y): #Using scikit learn
    start_time = datetime.now()
    # Initialize the model
    logreg = LogisticRegression(penalty=None, fit_intercept=False)

    # fit the model with data
    logreg.fit(data_x,data_y)
    betas = logreg.coef_.flatten() # flatten is To ensure it's a one dimensional array of d elements

    loss = log_loss(betas, data_x, data_y)
    end_time = datetime.now()
    return{
        "runtime" : (end_time - start_time).total_seconds(),
        "Total_Loss" : loss,
        "Batch_estimate": betas
    }

## Ridge Logistic Regression

In [53]:
def ridge_logistic_regression(data_x,data_y): #Using scikit learn
    start_time = datetime.now()
    # Initialize the model
    logreg = LogisticRegression(penalty="l2", random_state=1, fit_intercept=False)

    # fit the model with data
    logreg.fit(data_x,data_y)
    betas = logreg.coef_.flatten() # flatten is To ensure it's a one dimensional array of d elements

    loss = log_loss(betas, data_x, data_y)
    end_time = datetime.now()
    return{
        "runtime" : (end_time - start_time).total_seconds(),
        "Total_Loss" : loss,
        "Batch_estimate": betas
    }

# Testing algorithm functions

## Test performance of algorithms: runtime and loss

In [ ]:
@dataclass
class SimulationConfig:
    # Default is well specified
    runs: int = 5
    n: int = 200
    d: int = 3
    betas: list = field(default_factory=lambda: [3, 0.1, 5]) # This is to create a new list every time.
    x_means: list = field(default_factory=lambda: [2,-1,4])
    B: int = 40
    X: int = 10
    test_ratio: float = 0.3
    minimize_method: str = "l-bfgs-b"

In [86]:
def run_simulation(cfg: SimulationConfig):
    runs = cfg.runs
    n = cfg.n
    d = cfg.d
    betas = cfg.betas
    x_means = cfg.x_means
    B = cfg.B
    X = cfg.X
    test_ratio = cfg.test_ratio
    minimize_method = cfg.minimize_method

    results_all = []
    test_set_results = []

    for i in range(runs):

        test_size = round(n * test_ratio)
        train_size = n - test_size

        x = generate_x(n, d, x_means)
        y = generate_y(n, x, betas)

        # Split train/test
        x_train, y_train, x_test, y_test = x[:train_size], y[:train_size], x[train_size:], y[train_size:]

        # Run algorithms
        ogd = online_gd(n=train_size, d=d, data_y=y_train, data_x=x_train, B=B)
        aioli_model = AIOLI()
        ai = aioli_model.fit(data_y=y_train, data_x=x_train, X=X, B=B, minimize_method = minimize_method)
        classic_lg = classic_logistic_regression(data_x=x_train, data_y=y_train)
        ridge_lg = ridge_logistic_regression(data_x=x_train, data_y=y_train)

        # OTB estimates
        OGD_OTB_uniform = calculate_OTB(ogd["All_Betas"], "uniform")
        OGD_OTB_Suffix  = calculate_OTB(ogd["All_Betas"], "suffix_averaging")
        OGD_OTB_lastit  = calculate_OTB(ogd["All_Betas"], "last_iterate")

        # Training losses
        loss_OGD_Suffix = log_loss(OGD_OTB_Suffix, x_train, y_train)
        loss_OGD_lastit = log_loss(OGD_OTB_lastit, x_train, y_train)

        # TRAINING RESULTS

        algorithms_train = {
            "OGD_uniform": {
                "runtime": ogd["runtime"],
                "loss (trainset)": ogd["Total_Loss"],
                "online_loss": ogd["Online_Loss"],
                "betas": ogd["Batch_estimate"]
            },
            "OGD_Suffix": {
                "loss (trainset)": loss_OGD_Suffix,
                "betas": OGD_OTB_Suffix
            },
            "OGD_Last_iterate": {
                "loss (trainset)": loss_OGD_lastit,
                "betas": OGD_OTB_lastit
            },
            "AIOLI_uniform": {
                "runtime": ai["runtime"],
                "loss (trainset)": ai["Total_Loss"],
                "online_loss": ai["Online_Loss"],
                "betas": ai["Batch_estimate"],
                "convergence": ai["convergence"],
                "minimizer_gradient": ai["minimizer_gradient"]
            },
            "Classical_LG": {
                "runtime": classic_lg["runtime"],
                "loss (trainset)": classic_lg["Total_Loss"],
                "betas": classic_lg["Batch_estimate"]
            },
            "Ridge_LG": {
                "runtime": ridge_lg["runtime"],
                "loss (trainset)": ridge_lg["Total_Loss"],
                "betas": ridge_lg["Batch_estimate"]
            }
        }

        for name, metrics in algorithms_train.items():
            results_all.append({"algorithm": name, **metrics})

        # TEST RESULTS

        loss_OGD_uniform_test = log_loss(OGD_OTB_uniform, x_test, y_test)
        loss_OGD_Suffix_test  = log_loss(OGD_OTB_Suffix,  x_test, y_test)
        loss_OGD_lastit_test  = log_loss(OGD_OTB_lastit,  x_test, y_test)

        loss_AIOLI_uniform_test = aioli_model.new_data_loss(new_data_x=x_test, new_data_y=y_test, OTB_method="uniform")["loss"]
        loss_AIOLI_Suffix_test = aioli_model.new_data_loss(new_data_x=x_test, new_data_y=y_test, OTB_method="suffix_averaging")["loss"]
        loss_AIOLI_lastit_test = aioli_model.new_data_loss(new_data_x=x_test, new_data_y=y_test, OTB_method="last_iterate")["loss"]

        loss_LG_test = log_loss(classic_lg["Batch_estimate"], x_test, y_test)
        loss_LG_Ridge_test = log_loss(ridge_lg["Batch_estimate"],   x_test, y_test)

        algorithms_test = {
            "OGD_uniform": {"loss (testset)": loss_OGD_uniform_test},
            "OGD_Suffix":  {"loss (testset)": loss_OGD_Suffix_test},
            "OGD_Last_iterate": {"loss (testset)": loss_OGD_lastit_test},
            "AIOLI_uniform": {"loss (testset)": loss_AIOLI_uniform_test},
            "AIOLI_Suffix":  {"loss (testset)": loss_AIOLI_Suffix_test},
            "AIOLI_lastit":  {"loss (testset)": loss_AIOLI_lastit_test},
            "Classical_LG":  {"loss (testset)": loss_LG_test},
            "Ridge_LG":      {"loss (testset)": loss_LG_Ridge_test}
        }

        for name, metrics in algorithms_test.items():
            test_set_results.append({"algorithm": name, **metrics})

    return {
        "final_results": pd.DataFrame(results_all),
        "test_results": pd.DataFrame(test_set_results)
    }


In [84]:
def print_results(results):
    summary_train = results["final_results"].groupby("algorithm")[["runtime", "loss (trainset)", "online_loss", "betas"]].mean()
    summary_train["betas"] = summary_train["betas"].apply(lambda array: np.round(array, 3))
    summary_test = results["test_results"].groupby("algorithm")[["loss (testset)"]].mean()
    display(summary_train)
    display(summary_test)

# Run different settings

In [93]:
well_specified = SimulationConfig(
    betas=[3,0.1,5],
    x_means=[2,-1,4],
    x_cov=[[1,0,0],[0,1,0],[0,0,1]]
)

well_specified_result = run_simulation(well_specified)

KeyboardInterrupt: 

In [85]:
print_results(well_specified)

,runtime,loss (trainset),online_loss,betas
algorithm,,,,
AIOLI_uniform,0.415723,79.721054,11.047511,"[47.06, -14.501, 32.559]"
Classical_LG,0.002954,0.828323,NaN,"[10.611, -0.217, 10.394]"
OGD_Last_iterate,NaN,5.864031,NaN,"[0.634, 0.526, 1.16]"
OGD_Suffix,NaN,6.121145,NaN,"[0.606, 0.514, 1.12]"
OGD_uniform,0.001745,6.610403,7.533347,"[0.559, 0.486, 1.044]"
Ridge_LG,0.001536,3.882590,NaN,"[1.129, 0.434, 1.563]"


,loss (testset)
algorithm,
AIOLI_Suffix,2.734658
AIOLI_lastit,3.199333
AIOLI_uniform,2.806280
Classical_LG,7.951279
OGD_Last_iterate,3.306062
OGD_Suffix,3.367772
OGD_uniform,3.463453
Ridge_LG,2.840426


In [ ]:
#final_results[final_results["algorithm"] == "AIOLI_uniform"]

In [64]:
summary = final_results.groupby("algorithm")[["runtime", "loss (trainset)", "online_loss", "betas"]].mean()
summary["betas"] = summary["betas"].apply(lambda array: np.round(array, 3)) # To round betas
summary

,runtime,loss (trainset),online_loss,betas
algorithm,,,,
AIOLI_uniform,0.469931,214.226472,189.139986,"[-25.492, 27.824, 2.331]"
Classical_LG,0.006442,22.000126,NaN,"[-1.714, 1.841, 0.127]"
OGD_Last_iterate,NaN,43.203970,NaN,"[-0.298, 0.473, 0.175]"
OGD_Suffix,NaN,44.301253,NaN,"[-0.225, 0.452, 0.227]"
OGD_uniform,0.002556,51.091097,54.148647,"[-0.122, 0.414, 0.292]"
Ridge_LG,0.004255,22.395387,NaN,"[-1.44, 1.56, 0.12]"


In [65]:
summary_test = test_results.groupby("algorithm")[["loss (testset)"]].mean()
summary_test

,loss (testset)
algorithm,
AIOLI_Suffix,85.567944
AIOLI_lastit,128.136806
AIOLI_uniform,103.846959
Classical_LG,14.215541
OGD_Last_iterate,20.338974
OGD_Suffix,20.913811
OGD_uniform,24.039348
Ridge_LG,13.604853


### Check AIOLI convergence

In [ ]:
round((final_results[final_results["algorithm"] == "AIOLI_uniform"]["convergence"].mean() * 100 / n).item(),2)

In [ ]:
round((final_results[final_results["algorithm"] == "AIOLI_uniform"]["minimizer_gradient"].mean() * 100 / n).item(),2)

# Ignore from here on, it's simply code to test some functions/debugging

## Tests

In [ ]:
#etas = np.array([1/np.sqrt(3),1/np.sqrt(3),1/np.sqrt(3)])
#xt = np.array([1,2,3])
#betas = np.array([[2,2,5],[1,1,3],[4,5,6]])
#reg_lambda = 0.3
#gradients = np.array([[2,2,5],[1,1,3],[4,5,6]])
#b = np.array([1,2,3])
#sum_loss = 5


#(sum(etas)/2)*np.inner((b-betas[2]),gradients[2])*np.inner(gradients[2],(b-betas[2]))
#np.linalg.norm(etas)


In [ ]:
#diff = b - betas
#inner_prod = np.sum(gradients * diff,axis = 1)
#np.sum(np.sum(etas)/2 * inner_prod**2)

In [ ]:
# x0 = np.array([0,0,0])

#res = minimize(parameter_function_AIOLI, x0, method = "L-BFGS-B", args=(xt, reg_lambda, sum_loss,first_sum, etas_gradients))
#parameter_function_AIOLI(x0, xt, betas, etas, reg_lambda, sum_loss,gradients)
#res.x

In [ ]:
# diff = b - betas
# gs_bs = np.sum(gradients * diff,axis = 1)
# quadratic = np.sum(np.sum(etas)/2 * gs_bs**2)
#
# l_hat = sum_loss + np.sum(gs_bs) + quadratic
#
# l_hat + np.log(1+np.exp(np.inner(b,xt))) + np.log(1+np.exp(np.inner(-b,xt))) + reg_lambda*np.linalg.norm(b)

## Generate data

In [ ]:
# n = 1000
#
# mean = [20, 5, 2]
# cov = [[1, 0, 0], [0, 3, 0], [0,0,2]]  # diagonal covariance
# d = np.shape(cov)[1]
# y = np.random.binomial(n=1, p=0.3, size=n)
#
# x1, x2, x3 = np.random.multivariate_normal(mean, cov, size = n).T
# plt.plot(x1, x2, 'x')
# plt.axis('equal')
# plt.show()
#
# plt.plot(x1, x3, 'x')
# plt.axis('equal')
# plt.show()


In [ ]:
# x = np.random.multivariate_normal(mean, cov, size = n)
# sns.countplot(x=y)
# plt.show()


In [ ]:
# np.log(1000)

## OTB conversion proof tests

In [ ]:
# weights = np.array([0.3,0.7])
# values = np.array([2,3])
#
# print(np.sum(values))
# print(np.sum(weights*values))